# Entropy-Based Diagnosis of Sampling Bias in Archaeological Predictive Modeling

Reproduces every figure, table and number in the manuscript. Figures render
inline and are written to `figures/`; tables render as dataframes and are
written to `results/`.

**Data.** The published supplementary dataset of Yaworsky et al. (2020) for the
Grand Staircase--Escalante National Monument, archived at tDAR
(doi:10.6067/XCV8457626), in `data/`. It contains 11,814 records with a binary
presence indicator and ten environmental covariates, **and no coordinates** —
site locations in the United States are withheld under statutory protection.
Every entropy quantity below is therefore defined over covariate space, not
geographic space.

**Running it.** Execute the cells in order. Sections 5 to 9 are the slow ones
and print progress; the whole notebook takes roughly 25 minutes.

## 1. Setup

In [ ]:
%matplotlib inline
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings('ignore', category=UserWarning)   # MKL/KMeans notice on Windows

from src.entropy import load, coverage_profile, presence_weights, FEATURES, SEED
from src.models import MODELS, fit_bayes
from src.evaluation import run_cv, k_sensitivity, out_of_fold, paired_summary, summary_table
from src import figures as F

F.use_style()
pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 40)

RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
K_MAIN, K_GRID = 100, [25, 50, 100, 200, 400]
print('setup complete')

## 2. Environment

The sentence printed here goes into the Software subsection of the manuscript.
The versions reported in the paper must be the versions that produced the
numbers, so copy it from this output rather than typing it.

In [ ]:
import platform, sklearn, scipy, matplotlib

ENV = pd.Series({
    'Python':       platform.python_version(),
    'scikit-learn': sklearn.__version__,
    'numpy':        np.__version__,
    'scipy':        scipy.__version__,
    'pandas':       pd.__version__,
    'matplotlib':   matplotlib.__version__,
}, name='version').to_frame()
display(ENV)

print('--- paste into the manuscript ---\n')
for line in [
    "Analyses were carried out in Python " + platform.python_version() + ". Model fitting,",
    "clustering, and cross-validation used \\texttt{scikit-learn} " + sklearn.__version__ + ";",
    "numerical work used \\texttt{numpy} " + np.__version__ + " and \\texttt{scipy}",
    scipy.__version__ + "; data handling used \\texttt{pandas} " + pd.__version__ + "; and figures",
    "were produced with \\texttt{matplotlib} " + matplotlib.__version__ + ".",
]:
    print(line)

## 3. Data

Table 1 of the manuscript.

In [ ]:
df, X, y, groups = load()

table1 = pd.DataFrame({
    'Quantity': ['Total records ($N$)',
                 'Presence records ($N_s$)',
                 'Background records ($N_b$)',
                 'Sample prevalence',
                 'Environmental predictors',
                 'Locational information',
                 'Entropy cells ($K$, default)',
                 'Cell construction',
                 'Cross-validation',
                 'Random seed'],
    'Value': [f'{len(y):,}',
              f'{int((y == 1).sum()):,}',
              f'{int((y == 0).sum()):,}',
              f'{y.mean():.3f}',
              str(len(FEATURES)),
              'none published',
              f'{K_MAIN}, varied over {K_GRID[0]}\u2013{K_GRID[-1]}',
              '$k$-means partition of the standardised covariate space',
              '10-fold stratified; 5-fold watershed-blocked',
              str(SEED)],
})
table1.to_csv(RESULTS / 'table1_data_summary.csv', index=False)
display(table1.style.hide(axis='index'))

print(f'distinct watershed values (blocking units): {len(np.unique(groups))}')
display(df[FEATURES].describe().T.round(3))

## 4. Coverage of covariate space — Figure 1

The diagnostic. Presence records are compared against background records over
the same partition, at five resolutions.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
X_scaled = MinMaxScaler().fit_transform(X)

coverage = coverage_profile(X_scaled, y, K_GRID)
coverage.to_csv(RESULTS / 'coverage_entropy.csv', index=False)
display(coverage.round(4).style.hide(axis='index'))

fig = F.fig_coverage(coverage)
F.save(fig, 'fig1_coverage_entropy')

## 5. The weighting function — Figures 2 and 3

Panel (a) of Figure 2 should cross $w=1$ exactly at $p_i = 1/K$, and the mean
realised weight should be 1.000. Figure 3 locates the over- and
under-represented cells; it is a projection of environmental variables, not a
map.

In [ ]:
w_main, km_main, w_cell_main, H_main = presence_weights(X_scaled, y, K_MAIN)
realised = w_main[y == 1]

display(pd.Series({
    'K': K_MAIN,
    'H(P)': round(H_main, 4),
    'weight min': round(float(w_cell_main.min()), 4),
    'weight max': round(float(w_cell_main.max()), 4),
    'mean weight over presences': round(float(realised.mean()), 4),
    'records with zero weight': int((realised == 0).sum()),
}, name='value').to_frame())

fig = F.fig_weight_function(X_scaled, y, K_MAIN)
F.save(fig, 'fig2_weight_function')

In [ ]:
fig = F.fig_entropy_cells_pca(X_scaled, y, K_MAIN)
F.save(fig, 'fig3_entropy_cells_pca')

## 6. Random cross-validation

**Slow, roughly 8 minutes.** Ten stratified folds, four frameworks, weighted and
unweighted. The entropy partition and the weights are fitted inside each
training fold.

In [ ]:
cv_random = run_cv(X, y, groups, scheme='stratified', n_splits=10, K=K_MAIN)
cv_random.to_csv(RESULTS / 'cv_stratified.csv', index=False)
display(summary_table(cv_random))

## 7. Watershed-blocked cross-validation

**Slow, roughly 4 minutes.** Folds are blocked by catchment, so no catchment
appears on both sides of a split.

In [ ]:
cv_blocked = run_cv(X, y, groups, scheme='grouped', n_splits=5, K=K_MAIN)
cv_blocked.to_csv(RESULTS / 'cv_grouped.csv', index=False)
display(summary_table(cv_blocked))

### Table 3 of the manuscript

Both schemes side by side.

In [ ]:
table3 = (pd.concat([cv_random, cv_blocked])
            .groupby(['scheme', 'model', 'weighting'])[['AUC', 'TSS', 'Kappa', 'Brier']]
            .agg(['mean', 'std']).round(4))
table3.to_csv(RESULTS / 'table3_performance.csv')
display(table3)

## 8. Figures 4 and 6

Figure 4 is the fold-by-fold comparison of weighted against unweighted.
Figure 6 compares the two validation designs.

In [ ]:
fig = F.fig_paired_auc(cv_random)
F.save(fig, 'fig4_paired_auc')

In [ ]:
fig = F.fig_blocked_cv(cv_random, cv_blocked)
F.save(fig, 'fig6_blocked_cv')

## 9. Resolution sensitivity — Figure 5

**Slow, roughly 6 minutes.** Does the conclusion depend on the number of cells?

In [ ]:
ks = k_sensitivity(X, y, K_GRID, n_splits=5)
ks.to_csv(RESULTS / 'k_sensitivity.csv', index=False)
display(ks.groupby(['K', 'weighting'])[['AUC', 'TSS', 'H', 'w_max']].mean().round(4))

fig = F.fig_k_sensitivity(ks)
F.save(fig, 'fig5_k_sensitivity')

## 10. Calibration and variable importance — Figures 7 and 8

**Slow, roughly 5 minutes.** Out-of-fold predictions and held-out permutation
importance.

In [ ]:
oof, imp = out_of_fold(X, y, K=K_MAIN, n_splits=5)
np.savez(RESULTS / 'out_of_fold.npz', y=y,
         oof_unweighted=oof['unweighted'], oof_entropy=oof['entropy'],
         imp_unweighted=imp['unweighted'], imp_entropy=imp['entropy'],
         features=np.array(FEATURES))

from sklearn.metrics import roc_auc_score, brier_score_loss
display(pd.DataFrame([
    {'weighting': t, 'OOF AUC': round(roc_auc_score(y, oof[t]), 4),
     'OOF Brier': round(brier_score_loss(y, oof[t]), 4)}
    for t in ('unweighted', 'entropy')
]).style.hide(axis='index'))

fig = F.fig_calibration(y, oof)
F.save(fig, 'fig7_calibration')

In [ ]:
fig = F.fig_importance(imp, FEATURES)
F.save(fig, 'fig8_feature_importance')

display(pd.DataFrame({
    'variable': FEATURES,
    'unweighted': imp['unweighted'].mean(0).round(4),
    'entropy': imp['entropy'].mean(0).round(4),
}).sort_values('unweighted', ascending=False).style.hide(axis='index'))

## 11. Posterior for the Bayesian model — Figure 9

**Slow, roughly 1 minute.** Real MCMC draws, not a simulation.

In [ ]:
fig, posterior = F.fig_posterior(X_scaled, y, w_main, fit_bayes)
F.save(fig, 'fig9_bayesian_posterior')

display(pd.DataFrame([
    {'weighting': t,
     'acceptance rate': round(posterior[t]['acceptance'], 3),
     'median posterior variance': f"{np.median(posterior[t]['var']):.2e}"}
    for t in ('unweighted', 'entropy')
]).style.hide(axis='index'))

## 12. Paired tests

The differences and confidence intervals quoted in the Results.

In [ ]:
paired = pd.concat([paired_summary(cv_random, m) for m in ('AUC', 'TSS', 'Brier')])
paired.to_csv(RESULTS / 'paired_tests.csv', index=False)
display(paired.round(4).style.hide(axis='index'))

## 13. Headline numbers

Everything quoted in the abstract and conclusions, in one place. Check these
against the manuscript before submitting.

In [ ]:
k100 = coverage[coverage.K == 100].iloc[0]
rf_rand = cv_random[(cv_random.model == 'Random Forest') &
                    (cv_random.weighting == 'unweighted')].AUC.mean()
rf_block = cv_blocked[(cv_blocked.model == 'Random Forest') &
                      (cv_blocked.weighting == 'unweighted')].AUC.mean()
worst_rand = cv_random[cv_random.weighting == 'unweighted'] \
    .groupby('model').AUC.mean().min()
worst_block = cv_blocked[cv_blocked.weighting == 'unweighted'] \
    .groupby('model').AUC.mean().min()
auc_diffs = paired[paired.metric == 'AUC'].difference.abs()
ks_ent = ks[ks.weighting == 'entropy'].groupby('K').AUC.mean()

headline = pd.Series({
    'J presence, K=100':            round(k100.J_presence, 3),
    'J background, K=100':          round(k100.J_background, 3),
    'KL, K=25':                     round(coverage[coverage.K == 25].iloc[0].KL, 2),
    'KL, K=400':                    round(coverage[coverage.K == 400].iloc[0].KL, 2),
    'mean realised weight':         round(float(realised.mean()), 3),
    'largest |AUC difference|':     round(float(auc_diffs.max()), 4),
    'RF AUC, random CV':            round(rf_rand, 3),
    'RF AUC, blocked CV':           round(rf_block, 3),
    'best-worst margin, random':    round(rf_rand - worst_rand, 3),
    'best-worst margin, blocked':   round(rf_block - worst_block, 3),
    'K-sensitivity AUC range':      f'{ks_ent.min():.4f} to {ks_ent.max():.4f}',
}, name='value').to_frame()
headline.to_csv(RESULTS / 'headline_numbers.csv')
display(headline)

## 14. Files written

Figures in `figures/` as both PDF and PNG; tables and raw fold-level results in
`results/`.

In [ ]:
for folder in ('figures', 'results'):
    print(folder + '/')
    for f in sorted(Path(folder).iterdir()):
        print(f'  {f.name}')